# 03b — TabPFN, on Colab

The tabular foundation model of the comparison. TabPFN needs a GPU to be practical, so it lives here
rather than in `03_models.ipynb`.

Like the other two models, it predicts **the two decisions separately and multiplies them** rather
than predicting the match directly. Same folds, same target, so the three are comparable.

**How to run it**

1. run `03_models.ipynb` locally first — it writes `data/processed/model_table.csv`;
1. open this notebook in Colab and set *Runtime → Change runtime type → T4 GPU*;
1. add your TabPFN key as a Colab secret named `TABPFN_TOKEN` (key icon in the left bar, and enable
   notebook access). Never paste the key in a cell: it ends up in the file, and in git;
1. run the cells; the upload cell asks for `data/processed/model_table.csv`;
1. the last cell downloads `tabpfn_oof.csv` and `tabpfn_importance.csv`. Put both in `data/processed/`
   and run section 7 of `03_models.ipynb`.

The fold numbers travel inside the file, so TabPFN is scored on exactly the same split as the
logistic regression and XGBoost. Nothing here recomputes a split.

**What comes back**

- `tabpfn`: the match probability, her × his (the model we compare);
- `tabpfn_her`, `tabpfn_his`: the two decision probabilities, for the decision-level analysis;
- `tabpfn_direct`: a one-stage version fitted on the pairs, to test the two-stage trick on TabPFN too;
- `tabpfn_importance.csv`: permutation importance per fold (skip it with `IMPORTANCE = False`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install tabpfn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 46.4 MB/s eta 0:00:00


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, or expect about ten minutes a fold.")

GPU: Tesla T4


## 1. The data

In Colab this opens a file picker: choose `model_table.csv`. Outside Colab it reads the file
from the repository, so the notebook can also be run locally if the GPU is not available.

In [ ]:
import numpy as np
import pandas as pd

table = pd.read_csv("model_table.csv")

print(table.shape, "| folds:", sorted(table.fold.unique()))

(4184, 66) | folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 2. One row per decision

A match is she says yes and he says yes. Each pair becomes two rows — one per decider, who becomes
`self_` while the partner becomes `other_` — and the two predicted probabilities are multiplied back
into a match probability at the end.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

TARGETS = ["pair", "wave", "match", "fold", "her_dec", "his_dec"]
SIDE_CATEGORICAL = ["self_field", "self_race", "self_goal",
                    "other_field", "other_race", "other_goal"]
DIRECT_CATEGORICAL = ["her_field", "her_race", "her_goal",
                      "his_field", "his_race", "his_goal"]

def decision_view(tab, decider):
    mine, theirs = ("her", "his") if decider == "her" else ("his", "her")
    view = tab.drop(columns=TARGETS).rename(
        columns=lambda c: c.replace(f"{mine}_", "self_").replace(f"{theirs}_", "other_"))
    view["agediff"] = view.self_age - view.other_age   # directional: follows whoever is deciding
    view["female"] = int(decider == "her")
    return view

def preparation(categorical=SIDE_CATEGORICAL):
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), categorical)],
        remainder="passthrough")
    return make_pipeline(
      encode, SimpleImputer(strategy="median"))

hers, his = decision_view(table, "her"), decision_view(table, "his")
y = table.match
print(hers.shape, "rows per side ->", 2 * len(hers), "decisions")

(4184, 61) rows per side -> 8368 decisions


## 3. Fit and predict, fold by fold

TabPFN does not train in the usual sense: it carries the training rows and makes its predictions in a
single forward pass, so a fold takes seconds on a GPU. `n_estimators` is how many internal ensemble
members it averages. Note that a training fold now holds about 6,700 decisions rather than 3,300
pairs, which is still well inside what TabPFN handles.

Two settings worth knowing about, both tested:

- `ignore_pretraining_limits=True` is only needed off the GPU. TabPFN refuses to run on a CPU with
  more than 1,000 rows, and we have 3,337 per training fold. On a GPU the flag changes nothing; on a
  CPU it makes the notebook run, slowly — count roughly ten minutes a fold.
- if the GPU runs out of memory, lower `n_estimators` to 4 or 2 before anything else.

Two more things come out of the same loop. The two decision probabilities are kept separately, so the
analysis can look at her decision and his decision on their own. And, if `IMPORTANCE` is on, each fold
also gets a **permutation importance**: shuffle one group of columns in the held-out decisions and
measure how much the ROC-AUC falls. Columns are grouped by attribute (`self_age` and `other_age`
together, `agediff` with them), so a feature and its copies do not hide each other. That is one
prediction per group and per fold, so expect a few minutes per fold on a T4. Set `IMPORTANCE = False`
to skip it.


In [ ]:
import os
from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")

In [ ]:
import time
from sklearn.metrics import roc_auc_score
from tabpfn import TabPFNClassifier

IMPORTANCE = True   # permutation importance on every fold: a few minutes per fold on a T4. False skips it.

def feature_groups(columns, merge={"agediff": "age"}):
    """self_x and other_x form one group `x`; a derived column joins the group it comes from."""
    found = {}
    for c in columns:
        base = c.split("_", 1)[1] if c.startswith(("self_", "other_")) else c
        found.setdefault(base, []).append(c)
    for extra, into in merge.items():
        if extra in found and into in found:
            found[into] += found.pop(extra)
    return found

def group_importance(predict, frame, target, feature_sets, rng, repeats=1):
    """ROC-AUC lost when one group of columns is shuffled (every column of the group, same row order)."""
    base = roc_auc_score(target, predict(frame))
    drops = {}
    for name, cols in feature_sets.items():
        lost = []
        for _ in range(repeats):
            order = rng.permutation(len(frame))
            shuffled = frame.copy()
            for c in cols:
                shuffled[c] = frame[c].to_numpy()[order]
            lost.append(base - roc_auc_score(target, predict(shuffled)))
        drops[name] = float(np.mean(lost))
    return drops

feature_sets = feature_groups(hers.columns)
rng = np.random.default_rng(0)

predictions = pd.Series(np.nan, index=table.index)   # match probability = her x his
p_her = pd.Series(np.nan, index=table.index)
p_his = pd.Series(np.nan, index=table.index)
importance = []

for k in sorted(table.fold.unique()):
    train, test = table.fold != k, table.fold == k
    prep = preparation()

    started = time.time()
    both_sides = pd.concat([hers[train], his[train]])
    decisions = pd.concat([table.her_dec[train], table.his_dec[train]])

    model = TabPFNClassifier(n_estimators=8, random_state=0, ignore_pretraining_limits=True)
    model.fit(prep.fit_transform(both_sides), decisions)

    p_her[test] = model.predict_proba(prep.transform(hers[test]))[:, 1]
    p_his[test] = model.predict_proba(prep.transform(his[test]))[:, 1]
    predictions[test] = (p_her[test] * p_his[test]).to_numpy()
    print(f"fold {k}: {len(both_sides)} decisions / {test.sum()} pairs, {time.time() - started:.0f}s")

    if IMPORTANCE:
        started = time.time()
        frame = pd.concat([hers[test], his[test]], ignore_index=True)
        target = pd.concat([table.her_dec[test], table.his_dec[test]], ignore_index=True).to_numpy()
        drops = group_importance(lambda F: model.predict_proba(prep.transform(F))[:, 1],
                                 frame, target, feature_sets, rng)
        importance += [{"group": g, "fold": k, "drop": d} for g, d in drops.items()]
        print(f"        importance: {len(feature_sets)} groups, {time.time() - started:.0f}s")

print("missing predictions:", int(predictions.isna().sum()))

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

fold 0: 6674 decisions / 847 pairs, 52s
        importance: 37 groups, 729s
fold 1: 6614 decisions / 877 pairs, 38s
        importance: 37 groups, 716s
fold 2: 6714 decisions / 827 pairs, 38s
        importance: 37 groups, 735s
fold 3: 6722 decisions / 823 pairs, 39s
        importance: 37 groups, 728s
fold 4: 6748 decisions / 810 pairs, 39s
        importance: 37 groups, 737s
missing predictions: 0


### The one-stage baseline

The same model on the pairs directly: one row per pair, `match` as the target. This is what
`03_models.ipynb` compares against the two-stage version, as it does for the logit and XGBoost.

In [ ]:
X = table.drop(columns=TARGETS)
direct = pd.Series(np.nan, index=table.index)

for k in sorted(table.fold.unique()):
    train, test = table.fold != k, table.fold == k
    prep = preparation(DIRECT_CATEGORICAL)

    started = time.time()
    model = TabPFNClassifier(n_estimators=8, random_state=0, ignore_pretraining_limits=True)
    model.fit(prep.fit_transform(X[train]), y[train])
    direct[test] = model.predict_proba(prep.transform(X[test]))[:, 1]
    print(f"fold {k}: {train.sum()} pairs, {time.time() - started:.0f}s")

print("missing predictions:", int(direct.isna().sum()))

fold 0: 3337 pairs, 11s
fold 1: 3307 pairs, 11s
fold 2: 3357 pairs, 12s
fold 3: 3361 pairs, 11s
fold 4: 3374 pairs, 12s
missing predictions: 0


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

for label, p in (("two-stage", predictions), ("direct   ", direct)):
    print(f"{label}  ROC-AUC: {roc_auc_score(y, p):.3f}   PR-AUC: {average_precision_score(y, p):.3f}")

two-stage  ROC-AUC: 0.600   PR-AUC: 0.234
direct     ROC-AUC: 0.597   PR-AUC: 0.223


## 4. Send it back

`tabpfn_oof.csv` holds one row per pair, keyed on `pair` so `03_models.ipynb` can join it to the
other two models' predictions: the match probability, the two decision probabilities and the direct
version. `tabpfn_importance.csv` holds one row per feature group and fold.

In [ ]:
out = pd.DataFrame({"pair": table.pair, "tabpfn": predictions.to_numpy(), "tabpfn_direct": direct.to_numpy(),
                    "tabpfn_her": p_her.to_numpy(), "tabpfn_his": p_his.to_numpy()})
out.to_csv("tabpfn_oof.csv", index=False)
if importance:
    pd.DataFrame(importance).to_csv("tabpfn_importance.csv", index=False)

try:
    from google.colab import files
    files.download("tabpfn_oof.csv")
    if importance:
        files.download("tabpfn_importance.csv")
except ImportError:
    print("saved tabpfn_oof.csv (and tabpfn_importance.csv) next to this notebook")

out.head(3)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,pair,tabpfn,tabpfn_direct,tabpfn_her,tabpfn_his
0,"(1, 11)",0.094665,0.110331,0.304139,0.311256
1,"(1, 12)",0.274631,0.335615,0.579407,0.473986
2,"(1, 13)",0.207882,0.137808,0.392604,0.529495
